# Basic Information Retrieval System

**Course:** TECH 400 – Introduction to Information Retrieval
**Instructor:** Prof. Tek Raj Pant

This notebook implements a small, from-scratch Information Retrieval (IR) system. It covers:

1. Building a small document collection (7 documents)
2. Text pre-processing (tokenization, case-folding, stop-word removal)
3. Building the **dictionary** (vocabulary of unique terms)
4. Building the **inverted index** (term → list of document IDs)
5. Implementing **Boolean retrieval** functions (AND, OR, NOT)
6. Testing the system with sample queries

## 1. Import Libraries

We only need Python's built-in `re` module for simple text cleaning and `string` for punctuation handling. No external NLP libraries are required for a system this small, which keeps the implementation transparent and easy to follow.

In [1]:
import re
import string
from collections import defaultdict

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Document Collection

Below is our small collection of 7 short documents (IDs 1–7). Documents were chosen so that several key terms ("cat", "dog", "garden", "mat", "pet/pets") repeat across multiple documents. This overlap makes it possible to demonstrate meaningful **AND**, **OR**, and **NOT** Boolean queries later in the notebook.

In [2]:
documents = {
    1: "The cat sat on the mat and looked at the dog",
    2: "The dog chased the cat around the garden",
    3: "Cats and dogs are popular pets around the world",
    4: "The mat was made of soft cotton fabric",
    5: "Birds fly in the garden every morning",
    6: "The garden had colorful flowers and a small pond",
    7: "Pets like cats, dogs and birds require regular care",
}

for doc_id, text in documents.items():
    print(f"Doc {doc_id}: {text}")

Doc 1: The cat sat on the mat and looked at the dog
Doc 2: The dog chased the cat around the garden
Doc 3: Cats and dogs are popular pets around the world
Doc 4: The mat was made of soft cotton fabric
Doc 5: Birds fly in the garden every morning
Doc 6: The garden had colorful flowers and a small pond
Doc 7: Pets like cats, dogs and birds require regular care


## 3. Text Pre-processing

Before building the dictionary, every document must be normalized so that matching is consistent. The `preprocess()` function below performs three standard IR pre-processing steps:

1. **Case-folding** – convert all text to lowercase so that "Cat" and "cat" are treated as the same term.
2. **Punctuation removal** – strip commas, periods, and other punctuation marks.
3. **Tokenization** – split the cleaned text into individual word tokens.

A short list of common **stop words** (e.g., "the", "a", "and", "on") is also removed, since these words carry little discriminative meaning for retrieval and would otherwise appear in almost every posting list.

In [3]:
STOP_WORDS = {
    "a", "an", "the", "and", "or", "on", "at", "in", "of", "was",
    "were", "is", "are", "around", "like", "had"
}

def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS]
    return tokens

# Preview preprocessing on Document 1
print(preprocess(documents[1]))

['cat', 'sat', 'mat', 'looked', 'dog']


## 4. Building the Dictionary

The **dictionary** in an IR system is simply the sorted set of all unique, normalized terms that appear anywhere in the document collection. It is built by preprocessing every document and taking the union of all resulting tokens. The dictionary tells us the full vocabulary the system can search over — any query term that is not in the dictionary cannot match any document.

In [4]:
def build_dictionary(documents):
    vocabulary = set()
    for doc_id, text in documents.items():
        tokens = preprocess(text)
        vocabulary.update(tokens)
    return sorted(vocabulary)

dictionary = build_dictionary(documents)
print(f"Dictionary size: {len(dictionary)} terms\n")
print(dictionary)

Dictionary size: 27 terms

['birds', 'care', 'cat', 'cats', 'chased', 'colorful', 'cotton', 'dog', 'dogs', 'every', 'fabric', 'flowers', 'fly', 'garden', 'looked', 'made', 'mat', 'morning', 'pets', 'pond', 'popular', 'regular', 'require', 'sat', 'small', 'soft', 'world']


## 5. Building the Inverted Index

The **inverted index** is the core data structure of the IR system. For every term in the dictionary, it stores a sorted list (posting list) of the IDs of every document in which that term occurs. Instead of scanning every document for every query (a slow linear scan), the system can look up a term directly and instantly retrieve the set of documents that contain it. This is what makes Boolean retrieval fast even as the collection grows.

In [5]:
def build_inverted_index(documents):
    index = defaultdict(set)
    for doc_id, text in documents.items():
        tokens = preprocess(text)
        for token in tokens:
            index[token].add(doc_id)
    # store posting lists as sorted lists for readability
    return {term: sorted(doc_ids) for term, doc_ids in sorted(index.items())}

inverted_index = build_inverted_index(documents)
for term, postings in inverted_index.items():
    print(f"{term:12s} -> {postings}")

birds        -> [5, 7]
care         -> [7]
cat          -> [1, 2]
cats         -> [3, 7]
chased       -> [2]
colorful     -> [6]
cotton       -> [4]
dog          -> [1, 2]
dogs         -> [3, 7]
every        -> [5]
fabric       -> [4]
flowers      -> [6]
fly          -> [5]
garden       -> [2, 5, 6]
looked       -> [1]
made         -> [4]
mat          -> [1, 4]
morning      -> [5]
pets         -> [3, 7]
pond         -> [6]
popular      -> [3]
regular      -> [7]
require      -> [7]
sat          -> [1]
small        -> [6]
soft         -> [4]
world        -> [3]


## 6. Boolean Retrieval Functions

With the inverted index built, we can now implement the three classic Boolean operators:

- **AND** — returns documents that contain *both* terms (set intersection of the two posting lists).
- **OR** — returns documents that contain *either* term (set union of the two posting lists).
- **NOT** — returns documents that do **not** contain the given term (complement with respect to the full set of document IDs).

Each function looks up the relevant posting list(s) in the inverted index and applies the corresponding Python set operation. If a query term is not found in the index (i.e., it is not in the dictionary), an empty set is returned for that term instead of raising an error.

In [6]:
ALL_DOC_IDS = set(documents.keys())

def get_postings(term):
    """Return the posting list (as a set) for a single term."""
    term = term.lower()
    return set(inverted_index.get(term, []))

def boolean_and(term1, term2):
    return sorted(get_postings(term1) & get_postings(term2))

def boolean_or(term1, term2):
    return sorted(get_postings(term1) | get_postings(term2))

def boolean_not(term):
    return sorted(ALL_DOC_IDS - get_postings(term))

print("Functions defined: boolean_and, boolean_or, boolean_not")

Functions defined: boolean_and, boolean_or, boolean_not


## 7. Testing the Boolean Retrieval System

Finally, we test each of the three Boolean operators with example queries drawn from terms that appear in multiple documents, and print the resulting document IDs together with their original text so the results can be verified by eye.

In [7]:
def show_results(doc_ids, query_description):
    print(f"Query: {query_description}")
    print(f"Matching Document IDs: {doc_ids}")
    for doc_id in doc_ids:
        print(f"   Doc {doc_id}: {documents[doc_id]}")
    print()

# Test 1: AND query
result1 = boolean_and("cat", "dog")
show_results(result1, "cat AND dog")

# Test 2: OR query
result2 = boolean_or("garden", "mat")
show_results(result2, "garden OR mat")

# Test 3: NOT query
result3 = boolean_not("garden")
show_results(result3, "NOT garden")

Query: cat AND dog
Matching Document IDs: [1, 2]
   Doc 1: The cat sat on the mat and looked at the dog
   Doc 2: The dog chased the cat around the garden

Query: garden OR mat
Matching Document IDs: [1, 2, 4, 5, 6]
   Doc 1: The cat sat on the mat and looked at the dog
   Doc 2: The dog chased the cat around the garden
   Doc 4: The mat was made of soft cotton fabric
   Doc 5: Birds fly in the garden every morning
   Doc 6: The garden had colorful flowers and a small pond

Query: NOT garden
Matching Document IDs: [1, 3, 4, 7]
   Doc 1: The cat sat on the mat and looked at the dog
   Doc 3: Cats and dogs are popular pets around the world
   Doc 4: The mat was made of soft cotton fabric
   Doc 7: Pets like cats, dogs and birds require regular care



## 8. Conclusion

This notebook built a complete, minimal Information Retrieval pipeline: a small document collection, a normalized dictionary of terms, an inverted index mapping terms to document postings, and Boolean AND/OR/NOT retrieval functions built directly on top of that index. Testing confirmed that each operator returns the expected documents (e.g., `cat AND dog` correctly returns only the documents where both terms co-occur). This same design — normalize, index, then query with set operations — is the foundation that larger, real-world search engines build upon with additional techniques such as ranking, stemming, and phrase queries.